In [1]:
import json
from pathlib import Path
from datasets import Dataset

OUT = Path("results/advanced")
OUT.mkdir(parents=True, exist_ok=True)

# make_sft_dataset.py で作った用語集 Q&A をそのまま学習に使う
records = [
    json.loads(line)
    for line in Path("../glossary_sft.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
train_ds = Dataset.from_list(records)

# 評価用の質問（自作）と正解
#   known      … 用語集にある語（学習データと同じ言い回し）
#   paraphrase … 用語集にある語を言い回しを変えて質問
#   unknown    … 用語集にない語。正解は「用語集にない」と答えること
# keywords: 正解の回答に含まれるべき語
EVAL_SET = [
    {"type": "known", "question": "「LoRA」とは何ですか？", "keywords": ["低ランク"]},
    {"type": "known", "question": "「temperature」とは何ですか？", "keywords": ["ランダム"]},
    {"type": "known", "question": "「レート制限」とは何ですか？", "keywords": ["上限"]},
    {"type": "known", "question": "「過学習」とは何ですか？", "keywords": ["学習データ"]},

    {"type": "paraphrase", "question": "LoRAでは事前学習済みモデルのどの部分を学習しますか？", "keywords": ["低ランク"]},
    {"type": "paraphrase", "question": "temperatureの値を上げると出力はどう変わりますか？", "keywords": ["多様"]},
    {"type": "paraphrase", "question": "APIの呼び出し回数の上限を超えるとどうなりますか？", "keywords": ["429"]},
    {"type": "paraphrase", "question": "学習データに合わせすぎて新しい入力に弱くなる状態を何と呼びますか？", "keywords": ["過学習"]},

    {"type": "unknown", "question": "「RAG」とは何ですか？", "keywords": ["用語集にない"]},
    {"type": "unknown", "question": "ベクトルデータベースについて簡単に説明してください。", "keywords": ["用語集にない"]},
    {"type": "unknown", "question": "「プロンプトインジェクション」とは何ですか？", "keywords": ["用語集にない"]},
    {"type": "unknown", "question": "「RLHF」とは何ですか？", "keywords": ["用語集にない"]},
]

print(f"Train: {len(train_ds)}")
print(f"Eval: {len(EVAL_SET)}")

Train: 116
Eval: 12


/Users/julia/miniforge3/envs/matsuo-ai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen3.5-0.8B"
REVISION = "2fc06364715b967f1860aea9cf38778875588b17"

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

tok = AutoTokenizer.from_pretrained(BASE_MODEL, revision=REVISION)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=REVISION, dtype=torch.float32
).to(DEVICE)
model.eval()
print("Device:", DEVICE)


# 学習データと同じく、システムプロンプトなしで質問だけを渡す
def generate(model, question):
    prompt = tok.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=192, do_sample=False,
            pad_token_id=tok.pad_token_id,
        )
    return tok.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()


# keywords がすべて回答に含まれていれば正解
def run_eval(answer_fn):
    rows = []
    for item in EVAL_SET:
        answer = answer_fn(item["question"])
        rows.append({
            "type": item["type"],
            "question": item["question"],
            "answer": answer,
            "ok": all(k in answer for k in item["keywords"]),
        })
    df = pd.DataFrame(rows)
    print(df.groupby("type", sort=False)["ok"].mean())
    return df


start = time.time()
base_df = run_eval(lambda q: generate(model, q))
print(f"Evaluation time: {time.time() - start:.1f}s")
display(base_df)

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 425.80it/s]


Device: mps


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


type
known         0.0
paraphrase    0.5
unknown       0.0
Name: ok, dtype: float64
Evaluation time: 113.4s


,type,question,answer,ok
0,known,「LoRA」とは何ですか？,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つ...,False
1,known,「temperature」とは何ですか？,「temperature」は、**温度**（熱の強さや状態）を意味します。\n\nこの言葉は...,False
2,known,「レート制限」とは何ですか？,**レート制限**（Rate Limiting）とは、サーバー側やネットワーク側で、特定のユ...,False
3,known,「過学習」とは何ですか？,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデ...,False
4,paraphrase,LoRAでは事前学習済みモデルのどの部分を学習しますか？,LoRA（Low-Rank Adaptation）は、**事前学習済みモデルの「低秩部分（L...,False
5,paraphrase,temperatureの値を上げると出力はどう変わりますか？,温度の値を上げると、出力は**「温度が低い状態（冷たい状態）」**に変化します。\n\nこれ...,False
6,paraphrase,APIの呼び出し回数の上限を超えるとどうなりますか？,API の呼び出し回数の上限を超えると、**「オーバーフロー（Overflow）」**という...,True
7,paraphrase,学習データに合わせすぎて新しい入力に弱くなる状態を何と呼びますか？,学習データに合わせすぎて新しい入力に弱くなる状態を、**「過学習（Overfitting）」...,True
8,unknown,「RAG」とは何ですか？,**RAG** は、**R**eading **A**ll **G**enerated **...,False
9,unknown,ベクトルデータベースについて簡単に説明してください。,ベクトルデータベース（Vector Database）は、**「データが空間に位置している」...,False


In [3]:
base_df.to_json(OUT / "baseline.jsonl", orient="records", lines=True, force_ascii=False)

In [4]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

model.config.use_cache = False

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

sft = SFTConfig(
    output_dir=str(OUT / "ckpt"),
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=6,
    max_length=512,
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=10,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft,
    train_dataset=train_ds,
    peft_config=lora,
    processing_class=tok,
)
trainer.model.print_trainable_parameters()

start = time.time()
train_result = trainer.train()
training_seconds = time.time() - start

# training_loss は全ステップの平均。最後のステップの loss は log_history の最後
last_loss = [h["loss"] for h in trainer.state.log_history if "loss" in h][-1]
print(f"Training time: {training_seconds:.1f}s")
print(f"Last loss: {last_loss:.6f}  (mean: {train_result.training_loss:.6f})")

trainer.model.save_pretrained(OUT / "adapter")

tuned = trainer.model
tuned.config.use_cache = True
tuned.eval();

Truncating train dataset: 100%|██████████| 116/116 [00:00<00:00, 17563.33 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 116/116 [00:00<00:00, 49195.07 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


/Users/julia/miniforge3/envs/matsuo-ai/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.276012
20,2.212767
30,1.678353
40,0.983970
50,0.568986
60,0.343938
70,0.173698
80,0.149818
90,0.120207
100,0.099816


/Users/julia/miniforge3/envs/matsuo-ai/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/julia/miniforge3/envs/matsuo-ai/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/julia/miniforge3/envs/matsuo-ai/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/julia/miniforge3/envs/matsuo-ai/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/julia

Training time: 1089.1s
Last loss: 0.081674  (mean: 0.669360)


In [5]:
lora_df = run_eval(lambda q: generate(tuned, q))
display(lora_df)
lora_df.to_json(OUT / "after_lora.jsonl", orient="records", lines=True, force_ascii=False)

type
known         1.0
paraphrase    0.5
unknown       0.0
Name: ok, dtype: float64


,type,question,answer,ok
0,known,「LoRA」とは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,True
1,known,「temperature」とは何ですか？,次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる...,True
2,known,「レート制限」とは何ですか？,一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラ...,True
3,known,「過学習」とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,True
4,paraphrase,LoRAでは事前学習済みモデルのどの部分を学習しますか？,LoRAでは、重みを2つの小さな行列の積（低ランク行列）で表し、その2つの行列を独立に学習す...,True
5,paraphrase,temperatureの値を上げると出力はどう変わりますか？,次のトークンを選ぶときのランダムさを増やし、出力が少し変わる。設定ファイルの「tempera...,False
6,paraphrase,APIの呼び出し回数の上限を超えるとどうなりますか？,APIが有限にわるい。回数を増やしすぎるとレート制限が破られるか、レート制限の振る舞い（20...,True
7,paraphrase,学習データに合わせすぎて新しい入力に弱くなる状態を何と呼びますか？,過去の学習データにだけ過度に適合し、新しい入力に対して性能が落ちる状態。モデルが知らない入力...,False
8,unknown,「RAG」とは何ですか？,LLMが外部のデータやモデルを参照できるようにするシステム。モデルに直接接続する代替方法で、...,False
9,unknown,ベクトルデータベースについて簡単に説明してください。,大量のツールの呼び出しを同じ書き方で受け取る仕組み。Pydantic AIでは QueryG...,False


In [6]:
import re

# 用語集を {用語: 定義} の辞書にする
glossary = {}
for record in records:
    match = re.match(r"^「(.+?)」とは何ですか？$", record["messages"][0]["content"])
    if match:
        glossary[match.group(1)] = record["messages"][-1]["content"]
print("Glossary terms:", len(glossary))


# 検索: 質問文に含まれる用語名（括弧の前の部分）のうち、いちばん長いものを返す
def search_glossary(question):
    found = [
        term for term in glossary
        if term.split("（")[0] in question
    ]
    return max(found, key=lambda t: len(t.split("（")[0]), default=None)


for item in EVAL_SET:
    print(item["type"], item["question"], "->", search_glossary(item["question"]))

Glossary terms: 58
known 「LoRA」とは何ですか？ -> LoRA
known 「temperature」とは何ですか？ -> temperature
known 「レート制限」とは何ですか？ -> レート制限
known 「過学習」とは何ですか？ -> 過学習
paraphrase LoRAでは事前学習済みモデルのどの部分を学習しますか？ -> 事前学習
paraphrase temperatureの値を上げると出力はどう変わりますか？ -> temperature
paraphrase APIの呼び出し回数の上限を超えるとどうなりますか？ -> API
paraphrase 学習データに合わせすぎて新しい入力に弱くなる状態を何と呼びますか？ -> None
unknown 「RAG」とは何ですか？ -> None
unknown ベクトルデータベースについて簡単に説明してください。 -> None
unknown 「プロンプトインジェクション」とは何ですか？ -> プロンプト
unknown 「RLHF」とは何ですか？ -> None


In [7]:
# 用語集を検索して定義をプロンプトに入れ、元のモデル（LoRA なし）で答える
def answer_with_retrieval(question):
    term = search_glossary(question)
    if term is None:
        return "用語集にない語です。"
    prompt = (
        f"用語集: {term} … {glossary[term]}\n\n"
        f"この用語集に基づいて質問に答えてください。\n質問: {question}"
    )
    with tuned.disable_adapter():
        return generate(tuned, prompt)

In [8]:
retrieval_df = run_eval(answer_with_retrieval)
display(retrieval_df)
retrieval_df.to_json(OUT / "retrieval.jsonl", orient="records", lines=True, force_ascii=False)

type
known         0.75
paraphrase    0.25
unknown       0.75
Name: ok, dtype: float64


,type,question,answer,ok
0,known,「LoRA」とは何ですか？,LoRA（Low-Rank Adaptation）は、**事前学習済みの重み（パラメータ）を...,True
1,known,「temperature」とは何ですか？,temperature は、機械学習において**「温度（temperature）」**と呼ば...,False
2,known,「レート制限」とは何ですか？,「レート制限」とは、一定時間あたりに呼び出せるリクエスト数やトークン数の上限を指します。\n...,True
3,known,「過学習」とは何ですか？,「過学習」とは、学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態です。\n\...,True
4,paraphrase,LoRAでは事前学習済みモデルのどの部分を学習しますか？,LoRA（Low-Rank Adaptation）では、事前学習済みモデルの**ロマンタリン...,False
5,paraphrase,temperatureの値を上げると出力はどう変わりますか？,temperature の値を上げると、出力は**多様化（多様性）が増し、安定性が低下します...,True
6,paraphrase,APIの呼び出し回数の上限を超えるとどうなりますか？,API の呼び出し回数の上限を超えると、**API 呼び出しが失敗し、エラーが発生する**こ...,False
7,paraphrase,学習データに合わせすぎて新しい入力に弱くなる状態を何と呼びますか？,用語集にない語です。,False
8,unknown,「RAG」とは何ですか？,用語集にない語です。,True
9,unknown,ベクトルデータベースについて簡単に説明してください。,用語集にない語です。,True


In [9]:
summary = pd.DataFrame({
    "base": base_df.groupby("type", sort=False)["ok"].mean(),
    "lora": lora_df.groupby("type", sort=False)["ok"].mean(),
    "retrieval": retrieval_df.groupby("type", sort=False)["ok"].mean(),
})
summary.loc["all"] = [base_df["ok"].mean(), lora_df["ok"].mean(), retrieval_df["ok"].mean()]
display(summary)
summary.to_json(OUT / "summary.json", force_ascii=False, indent=2)

,base,lora,retrieval
type,,,
known,0.000000,1.0,0.750000
paraphrase,0.500000,0.5,0.250000
unknown,0.000000,0.0,0.750000
all,0.166667,0.5,0.583333


In [10]:
for i, item in enumerate(EVAL_SET):
    print(f"\n{'=' * 65}")
    print(f"[{item['type']}] {item['question']}  keywords={item['keywords']}")
    for name, df in [("BASE", base_df), ("LORA", lora_df), ("RETRIEVAL", retrieval_df)]:
        row = df.iloc[i]
        print(f"{name} ({'OK' if row['ok'] else 'NG'}):", row["answer"])


[known] 「LoRA」とは何ですか？  keywords=['低ランク']
BASE (NG): **LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。

簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。

具体的には、以下の 3 つのステップで考えられます：

1.  **ローカルモデルの作成**：
    研究者が、特定の分野（例：数学、言語、科学）に特化した、非常に小さなモデルを作成します。このモデルは、大規模モデルに比べて計算コストが非常に低く、学習速度も速いです。

2.  **LoRA の導入**：
    大規模モデル（例えば GPT-3.5 や Llama-3）にこのローカルモデルを「追加」して学習させます。この際、大規模モデルはロー
LORA (OK): 事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
RETRIEVAL (OK): LoRA（Low-Rank Adaptation）は、**事前学習済みの重み（パラメータ）を固定し、学習するパラメータのみを小さく学習する**、という手法です。

具体的には、以下の構成で動作します：
1.  **固定重み**: 学習対象のモデルの初期パラメータ（事前学習済み）を固定します。
2.  **低ランク行列**: 学習するパラメータを、2 つの小さな行列（低ランク行列）の積として表現します。
3.  **学習対象**: 学習するパラメータのみを、これらの低ランク行列を学習します。

このため、学習するパラメータの数が非常に少なくなり、学習結果（アダプタ）も非常に小さく保存できるため、**軽量なモデル**として非常に有用です。

[known] 「temperature」とは何ですか？  keywords=['ランダム']
BASE (NG): 「temperature」は、**温度**（熱の強さや状態）を意味します。

この言葉は英語の「temperature」